This notebook is my StreamGuard OTT project. The goal is to predict which streaming customers will cancel their subscription, and then use those predictions to estimate how much revenue is at risk. I’m using a synthetic Netflix-style churn dataset from Kaggle with 5,000 customers, so the results show what the models can learn from this data and not how they would do on real customers. Everything here is a notebook pipeline, not a deployed app.

In [1]:
import numpy as np
import pandas as pd


Downloading the dataset from Kaggle using kagglehub. I never edit this original file. All the cleaning later is done on copies of the training and test sets.

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abdulwadood11220/netflix-customer-churn-dataset")

print("Path to dataset files:", path)

100%|██████████| 186k/186k [00:00<00:00, 60.4MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/abdulwadood11220/netflix-customer-churn-dataset/versions/1


First look at the data. It has 5,000 rows and 14 columns, and there are no missing values at all. Churn is almost balanced, with 2,515 customers who churned and 2,485 who stayed, so accuracy is a fair metric here. customer_id is just an identifier so I won’t use it as a feature, and the six text columns will need encoding later.

In [5]:
import os,glob

csv_files = glob.glob(os.path.join(path, "*.csv"))
print(csv_files)

sg = pd.read_csv(csv_files[0])
print("Shape:", sg.shape)
print(sg.dtypes)
print(sg.head())
print("\nMissing values:")
print(sg.isnull().sum())
print("\nChurn counts:")
print(sg["churned"].value_counts())

['/root/.cache/kagglehub/datasets/abdulwadood11220/netflix-customer-churn-dataset/versions/1/netflix_customer_churn.csv']
Shape: (5000, 14)
customer_id                object
age                         int64
gender                     object
subscription_type          object
watch_hours               float64
last_login_days             int64
region                     object
device                     object
monthly_fee               float64
churned                     int64
payment_method             object
number_of_profiles          int64
avg_watch_time_per_day    float64
favorite_genre             object
dtype: object
                            customer_id  age  gender subscription_type  \
0  a9b75100-82a8-427a-a208-72f24052884a   51   Other             Basic   
1  49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b   47   Other          Standard   
2  4d71f6ce-fca9-4ff7-8afa-197ac24de14b   27  Female          Standard   
3  d3c72c38-631b-4f9e-8a0e-de103cad1a7d   53   Other           Premium   


Checking for bad values. There are no duplicate rows, no negative numbers, and the category spellings are clean. The categories are almost perfectly evenly split, which is another sign the data is synthetic. monthly_fee only has three values (8.99, 13.99 and 17.99), one for each plan. watch_hours is skewed to the right, with a median of 8.00 against a mean of 11.65. One column looks impossible: avg_watch_time_per_day goes up to 98.42 hours a day.

In [6]:
print(sg.describe().round(2).T)

print("\nDuplicate rows:", sg.duplicated().sum())

for col in sg.select_dtypes(include="object").columns:
    if sg[col].nunique() < 15:
        print("\n", col)
        print(sg[col].value_counts(dropna=False))

                         count   mean    std    min    25%    50%    75%  \
age                     5000.0  43.85  15.50  18.00  30.00  44.00  58.00   
watch_hours             5000.0  11.65  12.01   0.01   3.34   8.00  16.03   
last_login_days         5000.0  30.09  17.54   0.00  15.00  30.00  45.00   
monthly_fee             5000.0  13.68   3.69   8.99   8.99  13.99  17.99   
churned                 5000.0   0.50   0.50   0.00   0.00   1.00   1.00   
number_of_profiles      5000.0   3.02   1.42   1.00   2.00   3.00   4.00   
avg_watch_time_per_day  5000.0   0.87   2.62   0.00   0.11   0.29   0.72   

                           max  
age                      70.00  
watch_hours             110.40  
last_login_days          60.00  
monthly_fee              17.99  
churned                   1.00  
number_of_profiles        5.00  
avg_watch_time_per_day   98.42  

Duplicate rows: 0

 gender
gender
Female    1711
Male      1654
Other     1635
Name: count, dtype: int64

 subscription_type
s

I suspected avg_watch_time_per_day was just watch_hours divided by (last_login_days + 1). The biggest difference between that formula and the real column is 0.005, which is only rounding. So the column is derived from two others and adds no new information, and I’m dropping it.

In [7]:
check = sg["watch_hours"] / (sg["last_login_days"] + 1)
print((check - sg["avg_watch_time_per_day"]).abs().describe().round(3))

print(sg[["watch_hours", "last_login_days", "avg_watch_time_per_day"]].corr().round(2))

count    5000.000
mean        0.002
std         0.002
min         0.000
25%         0.001
50%         0.002
75%         0.004
max         0.005
dtype: float64
                        watch_hours  last_login_days  avg_watch_time_per_day
watch_hours                    1.00             0.00                    0.35
last_login_days                0.00             1.00                   -0.34
avg_watch_time_per_day         0.35            -0.34                    1.00



Outlier check using the IQR rule. Only watch_hours has real outliers, 238 customers or 4.8%. avg_watch_time_per_day is also flagged (549 rows, and 10 rows above 24 hours a day) but I’m removing that column anyway.

In [8]:
num_cols = ["age", "watch_hours", "last_login_days", "monthly_fee",
            "number_of_profiles", "avg_watch_time_per_day"]

for col in num_cols:
    Q1 = sg[col].quantile(0.25)
    Q3 = sg[col].quantile(0.75)
    IQR = Q3 - Q1
    low = Q1 - 1.5 * IQR
    high = Q3 + 1.5 * IQR
    n_out = ((sg[col] < low) | (sg[col] > high)).sum()
    print(f"{col:25s} bounds [{low:.2f}, {high:.2f}] | outliers: {n_out} ({n_out/len(sg)*100:.1f}%)")

print("\nRows with avg_watch_time_per_day above 24:", (sg["avg_watch_time_per_day"] > 24).sum())

age                       bounds [-12.00, 100.00] | outliers: 0 (0.0%)
watch_hours               bounds [-15.70, 35.07] | outliers: 238 (4.8%)
last_login_days           bounds [-30.00, 90.00] | outliers: 0 (0.0%)
monthly_fee               bounds [-4.51, 31.49] | outliers: 0 (0.0%)
number_of_profiles        bounds [-1.00, 7.00] | outliers: 0 (0.0%)
avg_watch_time_per_day    bounds [-0.81, 1.64] | outliers: 549 (11.0%)

Rows with avg_watch_time_per_day above 24: 10


Splitting the data. I left out customer_id (an identifier), avg_watch_time_per_day (redundant) and churned (the target). There are no duplicate customer IDs, so a normal random split is safe. I used a stratified 80/20 split, giving 4,000 training rows and 1,000 test rows, with a churn rate of 0.503 in both. I won’t touch the test set until the final model is chosen.

In [9]:
print("Duplicate customer_ids:", sg["customer_id"].duplicated().sum())

from sklearn.model_selection import train_test_split

X = sg.drop(columns=["customer_id", "avg_watch_time_per_day", "churned"])
y = sg["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn rate train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))
print(list(X.columns))

Duplicate customer_ids: 0
Train: (4000, 11) | Test: (1000, 11)
Churn rate train: 0.503 | test: 0.503
['age', 'gender', 'subscription_type', 'watch_hours', 'last_login_days', 'region', 'device', 'monthly_fee', 'payment_method', 'number_of_profiles', 'favorite_genre']


Capping watch_hours. I calculated the upper fence (35.09) from the training data only and applied it to both sets, so nothing from the test set leaks into the cleaning. I capped the extreme values instead of deleting them, which affected 178 training rows and 60 test rows, so I keep all 5,000 customers.


In [10]:
Q1 = X_train["watch_hours"].quantile(0.25)
Q3 = X_train["watch_hours"].quantile(0.75)
high = Q3 + 1.5 * (Q3 - Q1)
print("Upper fence from train:", round(high, 2))
print("Rows above fence | train:", (X_train["watch_hours"] > high).sum(),
      "| test:", (X_test["watch_hours"] > high).sum())

X_train = X_train.copy()
X_test = X_test.copy()
X_train["watch_hours"] = X_train["watch_hours"].clip(upper=high)
X_test["watch_hours"] = X_test["watch_hours"].clip(upper=high)

print("Max watch_hours after capping | train:", X_train["watch_hours"].max(),
      "| test:", X_test["watch_hours"].max())

Upper fence from train: 35.09
Rows above fence | train: 178 | test: 60
Max watch_hours after capping | train: 35.09125 | test: 35.09125


New feature: engagement_index is watch_hours divided by monthly_fee, so it means hours watched per dollar paid. It ranges from 0.001 to 3.9 with no infinite or missing values.



In [11]:
X_train["engagement_index"] = X_train["watch_hours"] / X_train["monthly_fee"]
X_test["engagement_index"] = X_test["watch_hours"] / X_test["monthly_fee"]

print(X_train["engagement_index"].describe().round(3))
print("Any infinite or missing values:",
      X_train["engagement_index"].isin([float("inf")]).sum(),
      X_train["engagement_index"].isnull().sum())

count    4000.000
mean        0.869
std         0.833
min         0.001
25%         0.242
50%         0.614
75%         1.234
max         3.903
Name: engagement_index, dtype: float64
Any infinite or missing values: 0 0


Building the preprocessing pipeline. Numeric columns get median imputation and then StandardScaler. Text columns get most-frequent imputation and then one-hot encoding, which gives 35 columns in total. Everything is fitted on the training data only and then applied to the test data. The imputers don’t change anything here because there are no blanks, but I kept them as a safeguard.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["age", "watch_hours", "last_login_days", "monthly_fee",
            "number_of_profiles", "engagement_index"]
cat_cols = ["gender", "subscription_type", "region", "device",
            "payment_method", "favorite_genre"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print("Train:", X_train_prep.shape, "| Test:", X_test_prep.shape)
print("Scaled train means (first 6):", X_train_prep[:, :6].mean(axis=0).round(2))
print("Scaled train stds  (first 6):", X_train_prep[:, :6].std(axis=0).round(2))
print(list(feature_names))

Train: (4000, 35) | Test: (1000, 35)
Scaled train means (first 6): [-0. -0.  0. -0.  0. -0.]
Scaled train stds  (first 6): [1. 1. 1. 1. 1. 1.]
['num__age', 'num__watch_hours', 'num__last_login_days', 'num__monthly_fee', 'num__number_of_profiles', 'num__engagement_index', 'cat__gender_Female', 'cat__gender_Male', 'cat__gender_Other', 'cat__subscription_type_Basic', 'cat__subscription_type_Premium', 'cat__subscription_type_Standard', 'cat__region_Africa', 'cat__region_Asia', 'cat__region_Europe', 'cat__region_North America', 'cat__region_Oceania', 'cat__region_South America', 'cat__device_Desktop', 'cat__device_Laptop', 'cat__device_Mobile', 'cat__device_TV', 'cat__device_Tablet', 'cat__payment_method_Credit Card', 'cat__payment_method_Crypto', 'cat__payment_method_Debit Card', 'cat__payment_method_Gift Card', 'cat__payment_method_PayPal', 'cat__favorite_genre_Action', 'cat__favorite_genre_Comedy', 'cat__favorite_genre_Documentary', 'cat__favorite_genre_Drama', 'cat__favorite_genre_Horro

Stratified 5-fold cross-validation, so every fold keeps about the same churn share. The always-predict-churn baseline gets accuracy 0.503, precision 0.503, recall 1.0, F1 0.669 and ROC-AUC 0.5. It flags everyone, so recall is perfect but half the flags are wrong. Logistic Regression gets accuracy 0.892, F1 0.892 and ROC-AUC 0.964.

In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

def cv_report(name, estimator):
    pipe = Pipeline([("prep", preprocessor), ("model", estimator)])
    res = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)
    print(f"{name:22s}", {m: round(res[f"test_{m}"].mean(), 3) for m in scoring})

cv_report("Baseline (majority)", DummyClassifier(strategy="most_frequent"))
cv_report("Logistic Regression", LogisticRegression(max_iter=1000))

Baseline (majority)    {'accuracy': np.float64(0.503), 'precision': np.float64(0.503), 'recall': np.float64(1.0), 'f1': np.float64(0.669), 'roc_auc': np.float64(0.5)}
Logistic Regression    {'accuracy': np.float64(0.892), 'precision': np.float64(0.893), 'recall': np.float64(0.892), 'f1': np.float64(0.892), 'roc_auc': np.float64(0.964)}


Recursive Feature Elimination keeps removing the weakest column. It kept 10 of the 35: watch_hours, last_login_days, monthly_fee, number_of_profiles, subscription_type_Standard and all five payment method columns. Inside cross-validation this smaller model scores F1 0.891 and ROC-AUC 0.964, the same as using all 35 columns, so the other 25 weren’t adding anything.

In [14]:
from sklearn.feature_selection import RFE

rfe = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=10, step=1)
rfe.fit(X_train_prep, y_train)

ranking = pd.Series(rfe.ranking_, index=feature_names).sort_values()
print("Selected by RFE (rank 1):")
print(list(ranking[ranking == 1].index))
print("\nEliminated last (rank 2 to 6):")
print(ranking[(ranking >= 2) & (ranking <= 6)])

rfe_pipe = Pipeline([
    ("prep", preprocessor),
    ("rfe", RFE(LogisticRegression(max_iter=1000), n_features_to_select=10, step=1)),
    ("model", LogisticRegression(max_iter=1000)),
])
res = cross_validate(rfe_pipe, X_train, y_train, cv=skf, scoring=scoring)
print("\nLogistic Regression + RFE (10 features):",
      {m: round(res[f"test_{m}"].mean(), 3) for m in scoring})

Selected by RFE (rank 1):
['num__watch_hours', 'num__last_login_days', 'num__monthly_fee', 'num__number_of_profiles', 'cat__subscription_type_Standard', 'cat__payment_method_Debit Card', 'cat__payment_method_Gift Card', 'cat__payment_method_Credit Card', 'cat__payment_method_Crypto', 'cat__payment_method_PayPal']

Eliminated last (rank 2 to 6):
cat__subscription_type_Basic       2
num__engagement_index              3
cat__favorite_genre_Drama          4
cat__favorite_genre_Documentary    5
cat__device_Laptop                 6
dtype: int64

Logistic Regression + RFE (10 features): {'accuracy': np.float64(0.89), 'precision': np.float64(0.892), 'recall': np.float64(0.89), 'f1': np.float64(0.891), 'roc_auc': np.float64(0.964)}


RFE keeping all five payment methods looked suspicious, so I checked. Churn rate by payment method runs from 0.430 for credit card to 0.595 for crypto, so payment method does carry some signal here. By plan, Basic churns at 0.607 while Standard and Premium are both around 0.451. I also removed columns one at a time with Logistic Regression and compared ROC-AUC: four numeric columns alone gave 0.955, adding payment method gave 0.962, everything except last_login_days gave 0.845, last_login_days alone gave 0.775 and watch_hours alone gave 0.808.

In [15]:
# Check 1: churn rate per category (training rows only)
for col in ["payment_method", "subscription_type"]:
    print(y_train.groupby(X_train[col]).mean().round(3))
    print()

# Check 2: ablation - which columns does the model actually need?
def cv_subset(name, num, cat):
    pre = ColumnTransformer([("num", numeric_pipe, num), ("cat", categorical_pipe, cat)])
    pipe = Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=1000))])
    res = cross_validate(pipe, X_train, y_train, cv=skf, scoring=["accuracy", "roc_auc"])
    print(f"{name:42s} acc {res['test_accuracy'].mean():.3f} | auc {res['test_roc_auc'].mean():.3f}")

four = ["watch_hours", "last_login_days", "monthly_fee", "number_of_profiles"]
cv_subset("4 numeric columns only", four, [])
cv_subset("4 numeric + payment_method", four, ["payment_method"])
cv_subset("all columns EXCEPT last_login_days",
          [c for c in num_cols if c != "last_login_days"], cat_cols)
cv_subset("last_login_days alone", ["last_login_days"], [])
cv_subset("watch_hours alone", ["watch_hours"], [])

payment_method
Credit Card    0.430
Crypto         0.595
Debit Card     0.440
Gift Card      0.579
PayPal         0.473
Name: churned, dtype: float64

subscription_type
Basic       0.607
Premium     0.452
Standard    0.451
Name: churned, dtype: float64

4 numeric columns only                     acc 0.881 | auc 0.955
4 numeric + payment_method                 acc 0.888 | auc 0.962
all columns EXCEPT last_login_days         acc 0.753 | auc 0.845
last_login_days alone                      acc 0.744 | auc 0.775
watch_hours alone                          acc 0.706 | auc 0.808


Comparing five models on the same folds without tuning. The F1 scores were 0.892 for Logistic Regression, 0.864 for KNN, 0.916 for SVM, 0.968 for Random Forest and 0.970 for AdaBoost. ROC-AUC went 0.964, 0.933, 0.976, 0.995 and 0.996 in the same order. The two ensemble models clearly beat the linear baseline, and Random Forest and AdaBoost are very close to each other.

In [16]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)": SVC(kernel="rbf", C=1.0),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
}

rows = {}
for name, est in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", est)])
    res = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)
    rows[name] = {m: res[f"test_{m}"].mean() for m in scoring}
    rows[name]["f1_std"] = res["test_f1"].std()

print(pd.DataFrame(rows).T.round(3))

                     accuracy  precision  recall     f1  roc_auc  f1_std
Logistic Regression     0.892      0.893   0.892  0.892    0.964   0.004
KNN (k=5)               0.861      0.852   0.878  0.864    0.933   0.009
SVM (RBF)               0.914      0.910   0.921  0.916    0.976   0.006
Random Forest           0.968      0.970   0.967  0.968    0.995   0.007
AdaBoost                0.970      0.974   0.966  0.970    0.996   0.008


A customer who already cancelled would naturally not have logged in recently, so last_login_days might partly reflect the outcome and not predict it. To see how much the models depend on it, I removed it. Random Forest dropped to F1 0.748 and ROC-AUC 0.862, and AdaBoost dropped to F1 0.761 and ROC-AUC 0.867. So most of the ensembles’ advantage comes from this one column.

In [17]:
def cv_without(name, est, drop_col):
    nums = [c for c in num_cols if c != drop_col]
    cats = [c for c in cat_cols if c != drop_col]
    pre = ColumnTransformer([("num", numeric_pipe, nums), ("cat", categorical_pipe, cats)])
    pipe = Pipeline([("prep", pre), ("model", est)])
    res = cross_validate(pipe, X_train, y_train, cv=skf, scoring=["f1", "roc_auc"])
    print(f"{name:34s} f1 {res['test_f1'].mean():.3f} | auc {res['test_roc_auc'].mean():.3f}")

cv_without("Random Forest, no last_login_days", models["Random Forest"], "last_login_days")
cv_without("AdaBoost, no last_login_days", models["AdaBoost"], "last_login_days")

Random Forest, no last_login_days  f1 0.748 | auc 0.862
AdaBoost, no last_login_days       f1 0.761 | auc 0.867


Tuning with RandomizedSearchCV, trying 15 random settings per model with 5-fold cross-validation scored on F1. Random Forest’s best CV F1 was 0.969. AdaBoost’s best was 0.987, using tree depth 3, 264 estimators and a learning rate of about 0.72. I picked AdaBoost from the cross-validation results alone, before looking at the test set.



In [18]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint, uniform

rf_pipe = Pipeline([("prep", preprocessor),
                    ("model", RandomForestClassifier(random_state=42))])
rf_space = {
    "model__n_estimators": randint(100, 400),
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": randint(2, 11),
    "model__max_features": ["sqrt", "log2"],
}

ada_pipe = Pipeline([("prep", preprocessor),
                     ("model", AdaBoostClassifier(estimator=DecisionTreeClassifier(), random_state=42))])
ada_space = {
    "model__n_estimators": randint(50, 300),
    "model__learning_rate": uniform(0.05, 1.5),
    "model__estimator__max_depth": [1, 2, 3],
}

searches = {}
for name, pipe, space in [("Random Forest", rf_pipe, rf_space), ("AdaBoost", ada_pipe, ada_space)]:
    s = RandomizedSearchCV(pipe, space, n_iter=15, cv=skf, scoring="f1",
                           random_state=42, n_jobs=-1)
    s.fit(X_train, y_train)
    searches[name] = s
    print(name, "| best CV F1:", round(s.best_score_, 3))
    print("  ", s.best_params_)

Random Forest | best CV F1: 0.969
   {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_split': 3, 'model__n_estimators': 188}
AdaBoost | best CV F1: 0.987
   {'model__estimator__max_depth': 3, 'model__learning_rate': np.float64(0.7187491292803867), 'model__n_estimators': 264}


Quick check that the tuned models are stored. I added this after the dictionary came back empty in an earlier run.If the AdaBoost search is missing, this re-runs it with the same settings and seed, which gives the same result as before (best CV F1 0.987).

In [21]:
print(list(searches.keys()))

[]


In [24]:
if "AdaBoost" not in searches:
    s = RandomizedSearchCV(ada_pipe, ada_space, n_iter=15, cv=skf, scoring="f1",
                           random_state=42, n_jobs=-1)
    s.fit(X_train, y_train)
    searches["AdaBoost"] = s

print("AdaBoost | best CV F1:", round(searches["AdaBoost"].best_score_, 3))
print(searches["AdaBoost"].best_params_)

AdaBoost | best CV F1: 0.987
{'model__estimator__max_depth': 3, 'model__learning_rate': np.float64(0.7187491292803867), 'model__n_estimators': 264}


Final evaluation on the 1,000 held-out customers, used only once. With all features, tuned AdaBoost got accuracy 0.990, F1 0.990 and ROC-AUC 0.999, with 6 false alarms and 4 missed churners. Without last_login_days it got accuracy 0.762, F1 0.764 and ROC-AUC 0.874, with 120 false alarms and 118 missed churners. I’m reporting both because it isn’t clear whether last_login_days would be known before a customer cancels. The near-perfect score comes from the data being synthetic, so it shouldn’t be read as real-world performance.

In [25]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.base import clone

best = searches["AdaBoost"].best_estimator_
pred = best.predict(X_test)
prob = best.predict_proba(X_test)[:, 1]
print("=== Tuned AdaBoost, all features, TEST set")
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))

nums_no_login = [c for c in num_cols if c != "last_login_days"]
pre_no_login = ColumnTransformer([("num", numeric_pipe, nums_no_login),
                                  ("cat", categorical_pipe, cat_cols)])
best_no_login = clone(best).set_params(prep=pre_no_login)
best_no_login.fit(X_train, y_train)
pred2 = best_no_login.predict(X_test)
prob2 = best_no_login.predict_proba(X_test)[:, 1]
print("\n=== Tuned AdaBoost, WITHOUT last_login_days, TEST set")
print(confusion_matrix(y_test, pred2))
print(classification_report(y_test, pred2, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, prob2), 3))

=== Tuned AdaBoost, all features, TEST set
[[491   6]
 [  4 499]]
              precision    recall  f1-score   support

           0      0.992     0.988     0.990       497
           1      0.988     0.992     0.990       503

    accuracy                          0.990      1000
   macro avg      0.990     0.990     0.990      1000
weighted avg      0.990     0.990     0.990      1000

ROC-AUC: 0.999

=== Tuned AdaBoost, WITHOUT last_login_days, TEST set
[[377 120]
 [118 385]]
              precision    recall  f1-score   support

           0      0.762     0.759     0.760       497
           1      0.762     0.765     0.764       503

    accuracy                          0.762      1000
   macro avg      0.762     0.762     0.762      1000
weighted avg      0.762     0.762     0.762      1000

ROC-AUC: 0.874


 Revenue at risk is the churn probability times the monthly fee, added up over customers. With all features the model expects $6,638 a month at risk, compared with $6,444 in fees actually paid by the customers who churned. Without last_login_days it expects $6,464. Premium has the most revenue at risk ($2,673) even though Basic churns the most, because Premium customers pay more. The model’s probabilities aren’t calibrated, so they’re better used to rank customers than as exact chances, and these totals are illustrations on past data, not forecasts.

In [26]:
def revenue_view(label, model):
    prob = model.predict_proba(X_test)[:, 1]
    out = pd.DataFrame({
        "customer_id": sg.loc[X_test.index, "customer_id"].values,
        "plan": X_test["subscription_type"].values,
        "monthly_fee": X_test["monthly_fee"].values,
        "churn_prob": prob,
        "actually_churned": y_test.values,
    })
    out["expected_at_risk"] = out["churn_prob"] * out["monthly_fee"]

    print(f"=== {label}")
    print("Expected monthly revenue at risk: $", round(out["expected_at_risk"].sum(), 2))
    print("Actual monthly fees of customers who churned: $",
          round(out.loc[out["actually_churned"] == 1, "monthly_fee"].sum(), 2))
    print("\nExpected at-risk revenue by plan:")
    print(out.groupby("plan")["expected_at_risk"].sum().round(2))
    print("\nTop 5 customers by at-risk revenue:")
    print(out.sort_values("expected_at_risk", ascending=False).head(5).round(3))
    print()
    return out

risk_full = revenue_view("All features", best)
risk_no_login = revenue_view("Without last_login_days", best_no_login)

=== All features
Expected monthly revenue at risk: $ 6638.19
Actual monthly fees of customers who churned: $ 6443.97

Expected at-risk revenue by plan:
plan
Basic       1731.38
Premium     2673.33
Standard    2233.48
Name: expected_at_risk, dtype: float64

Top 5 customers by at-risk revenue:
                              customer_id     plan  monthly_fee  churn_prob  \
861  57ffc492-dcb6-416b-a4f3-7109014eb262  Premium        17.99       0.813   
840  c51b4715-3ce4-49e7-9102-508107212399  Premium        17.99       0.800   
904  2cf69775-8761-45e2-a44d-b4bd0669b757  Premium        17.99       0.800   
597  7016829c-88f3-4821-9fc1-ce5343e53906  Premium        17.99       0.797   
575  c85a5890-a887-41bf-b866-5cd201747ede  Premium        17.99       0.796   

     actually_churned  expected_at_risk  
861                 1            14.620  
840                 1            14.399  
904                 1            14.388  
597                 1            14.333  
575                 1 

The dataset has no lifetime value column, so I made an estimate: monthly fee times 12 times (1 minus churn probability). The 12-month window and the risk tier cut-offs (Low up to 0.3, Medium 0.3 to 0.6, High above 0.6) are my own assumptions. The total 12-month CLV estimate is $84,406 and the total 12-month revenue exposed is $79,658. The real churn rate in each tier was 0.0% for Low (198 customers), 30.2% for Medium (427) and 99.7% for High (375), so the tiers rank customers in the right order. Ideas like contacting high-risk Premium customers first are only suggestions, since this data has no information about retention offers.

In [27]:
risk = risk_full.copy()
risk["clv_12m"] = risk["monthly_fee"] * 12 * (1 - risk["churn_prob"])
risk["revenue_exposed_12m"] = risk["monthly_fee"] * 12 * risk["churn_prob"]
risk["risk_tier"] = pd.cut(risk["churn_prob"], bins=[0, 0.3, 0.6, 1.0],
                           labels=["Low", "Medium", "High"], include_lowest=True)

print("Total 12-month CLV estimate: $", round(risk["clv_12m"].sum(), 2))
print("Total 12-month revenue exposed: $", round(risk["revenue_exposed_12m"].sum(), 2))

print("\nBy risk tier:")
print(risk.groupby("risk_tier", observed=True).agg(
    customers=("customer_id", "count"),
    actual_churn_rate=("actually_churned", "mean"),
    avg_clv_12m=("clv_12m", "mean"),
    exposed_12m=("revenue_exposed_12m", "sum"),
).round(3))

print("\nAverage 12-month CLV estimate by plan:")
print(risk.groupby("plan")["clv_12m"].mean().round(2))

Total 12-month CLV estimate: $ 84405.73
Total 12-month revenue exposed: $ 79658.27

By risk tier:
           customers  actual_churn_rate  avg_clv_12m  exposed_12m
risk_tier                                                        
Low              198              0.000      137.939     8208.305
Medium           427              0.302       94.761    31485.633
High             375              0.997       44.348    39964.332

Average 12-month CLV estimate by plan:
plan
Basic        45.67
Premium     120.97
Standard     86.17
Name: clv_12m, dtype: float64


Conclusion: tuned AdaBoost predicts churn very accurately on this synthetic dataset, with a test F1 of 0.990, but most of that comes from last_login_days. Without it the test F1 drops to 0.764. The main limitations are that the data is synthetic, there are no tenure or support columns, the probabilities aren’t calibrated, the CLV is an assumption-based estimate, and there is no evidence that any retention offer works. For future work I’d build a simple interface for viewing risk tiers, add a movie recommendation and watchlist module (not built yet) using a separate ratings dataset, use real customer data with tenure and support information, and run an experiment to test whether retention offers actually keep customers.